### 1. 경로 설정

In [1]:
import sys
from pathlib import Path

# ✅ 노트북 위치가 어디든 프로젝트 루트를 찾아 sys.path에 추가
ROOT = Path.cwd()
while (ROOT / "src").exists() is False and ROOT.parent != ROOT:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)

ROOT = c:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project


### 2. 평가 데이터셋 로드

In [2]:
import json

DATA_PATH = ROOT / "evaluation" / "test_questions.json"  # 본인 파일명으로
with open(DATA_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

len(dataset), dataset[0].keys()

(30, dict_keys(['id', 'document', 'question', 'answer', 'source_location']))

### 3. 평가용 retriever 함수

In [3]:
from src.retriever import rerank_docs

def retrieve_docs(question: str, k: int):
    # candidate_k는 rerank에 넣을 후보 수 (top_n보다 넉넉히)
    candidate_k = max(60, k * 3)
    return rerank_docs(
        question,
        top_n=k,                 # ✅ K개 반환
        candidate_k=candidate_k, # ✅ 후보는 넉넉히
        docid_scope_top_n=None,  # ✅ 평가에서는 필터 OFF
    )

### 4. Recall, MRR 계산 함수

In [4]:
import os, re
from typing import Any, List, Dict, Optional, Tuple

def _norm_fname(s: str) -> str:
    """basename + lower + 공백 정리"""
    if not s:
        return ""
    s = os.path.basename(str(s)).strip()
    s = re.sub(r"\s+", " ", s)
    return s.lower()

def _doc_fname(doc) -> str:
    md = getattr(doc, "metadata", None) or {}
    fn = md.get("file_name") or md.get("filename") or md.get("source") or md.get("path") or ""
    return _norm_fname(fn)

def _first_rank(docs: List[Any], gold_document: str, k: int) -> Optional[int]:
    """상위 k개에서 정답 문서가 최초 등장하는 rank(1-based). 없으면 None"""
    gold = _norm_fname(gold_document)
    for i, d in enumerate((docs or [])[:k], 1):
        if _doc_fname(d) == gold:
            return i
    return None

def recall_mrr_at_k(docs: List[Any], gold_document: str, k: int) -> Tuple[float, float]:
    r = _first_rank(docs, gold_document, k)
    if r is None:
        return 0.0, 0.0
    return 1.0, 1.0 / r

In [5]:
from typing import Callable

def evaluate_retrieval(
    dataset: List[Dict[str, Any]],
    retrieve_fn: Callable[[str, int], List[Any]],
    k_list: List[int] = [1, 3, 5, 10],
    debug_n: int = 0,   # 처음 N개만 검색 결과 출력 (0이면 미출력)
):
    n = len(dataset)
    recall_sum = {k: 0.0 for k in k_list}
    mrr_sum = {k: 0.0 for k in k_list}

    for idx, item in enumerate(dataset, 1):
        q = item["question"]
        gold_doc = item["document"]

        # ✅ 가장 큰 K로 한 번만 가져오고, 슬라이싱으로 재사용 (속도 개선)
        max_k = max(k_list)
        docs = retrieve_fn(q, max_k)

        for k in k_list:
            r, m = recall_mrr_at_k(docs, gold_doc, k)
            recall_sum[k] += r
            mrr_sum[k] += m

        if debug_n and idx <= debug_n:
            print(f"\n[DEBUG Q{idx:02d}] {q}")
            print("GOLD_DOC:", gold_doc)
            print("TOP docs:")
            for j, d in enumerate(docs[:min(5, len(docs))], 1):
                print(f"  {j}. {_doc_fname(d)}")

    results = {
        "n": n,
        "Recall": {k: recall_sum[k] / n for k in k_list},
        "MRR": {k: mrr_sum[k] / n for k in k_list},
    }
    return results

### 5. 평가

In [7]:
import pandas as pd

k_list = [1, 3, 5, 10, 20]  # 보고 싶은 K로 조정 (단, rerank_docs top_n이 이만큼 나와야 함)

results = evaluate_retrieval(
    dataset=dataset,
    retrieve_fn=retrieve_docs,
    k_list=k_list,
    debug_n=2,  # 처음 2개만 디버그 출력
)

df = pd.DataFrame({
    "K": list(results["Recall"].keys()),
    "Recall@K": list(results["Recall"].values()),
    "MRR@K": list(results["MRR"].values()),
})
df

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[DEBUG Q01] 이 사업의 예산(배정예산)은 총 얼마이며, 어떤 부서가 주관하나요?
GOLD_DOC: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
TOP docs:
  1. 국가철도공단_철도인프라 디지털트윈 정보화전략계획(isp) 수립 용역(변.hwp
  2. 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp
  3. 한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp
  4. 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp
  5. 수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ismp 수립 입.hwp


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



[DEBUG Q02] 사업 범위 중 '재난위험지역 경보발령 범위 확대'를 위해 사용되는 무선 통신 방식은?
GOLD_DOC: 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
TOP docs:
  1. 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp
  2. 경기도 안양시_호계체육관 배드민턴장 및 탁구장 예약시스템 구축 용역.hwp
  3. 한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp
  4. 경상북도 봉화군_봉화군 재난통합관리시스템 고도화 사업(협상)(긴급).hwp
  5. 서울특별시 여성가족재단_(재공고, 협상) 서울 디지털성범죄 안심지원센.hwp


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embedding

,K,Recall@K,MRR@K
0,1,0.066667,0.066667
1,3,0.066667,0.066667
2,5,0.133333,0.083333
3,10,0.200000,0.090000
4,20,0.266667,0.093627


### 6. BM25 / Dense(MMR) / Hybrid(RRF) / Rerank(FlashRank) 비교 평가

In [12]:
import sys, importlib
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.retriever as retriever
importlib.reload(retriever)
print("retriever file:", retriever.__file__)

retriever file: c:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project\src\retriever.py


In [9]:
import os, re
from typing import Any, List, Dict, Optional, Tuple, Callable
import pandas as pd

def norm_fname(s: str) -> str:
    if not s:
        return ""
    s = os.path.basename(str(s)).strip()
    s = re.sub(r"\s+", " ", s)
    return s.lower()

def doc_fname(doc) -> str:
    md = getattr(doc, "metadata", None) or {}
    fn = md.get("file_name") or md.get("filename") or md.get("source") or md.get("path") or ""
    return norm_fname(fn)

def first_rank(docs: List[Any], gold_document: str, k: int) -> Optional[int]:
    gold = norm_fname(gold_document)
    for i, d in enumerate((docs or [])[:k], 1):
        if doc_fname(d) == gold:
            return i
    return None

def recall_mrr_at_k(docs: List[Any], gold_document: str, k: int) -> Tuple[float, float]:
    r = first_rank(docs, gold_document, k)
    if r is None:
        return 0.0, 0.0
    return 1.0, 1.0 / r

def evaluate_retrieval(
    dataset: List[Dict[str, Any]],
    retrieve_maxk_fn: Callable[[str, int], List[Any]],
    k_list: List[int] = [1, 3, 5, 10, 20],
    debug_n: int = 0,
    label: str = "",
):
    n = len(dataset)
    max_k = max(k_list)
    recall_sum = {k: 0.0 for k in k_list}
    mrr_sum = {k: 0.0 for k in k_list}

    for idx, item in enumerate(dataset, 1):
        q = item["question"]
        gold_doc = item["document"]

        docs = retrieve_maxk_fn(q, max_k) or []

        for k in k_list:
            r, m = recall_mrr_at_k(docs, gold_doc, k)
            recall_sum[k] += r
            mrr_sum[k] += m

        if debug_n and idx <= debug_n:
            print(f"\n[DEBUG:{label} Q{idx:02d}] {q}")
            print("GOLD_DOC:", gold_doc)
            print("TOP5:")
            for j, d in enumerate(docs[:5], 1):
                print(f"  {j}. {doc_fname(d)}")

    return {
        "n": n,
        "Recall": {k: recall_sum[k] / n for k in k_list},
        "MRR": {k: mrr_sum[k] / n for k in k_list},
    }

### 7. 단계 별 함수

- BM25

In [13]:
def retrieve_bm25(question: str, k: int):
    r = retriever.get_bm25_retriever()
    docs = r.invoke(question) or []
    return docs[:k]

- Dense

In [14]:
def retrieve_dense(question: str, k: int):
    r = retriever.get_dense_retriever()
    docs = r.invoke(question) or []
    return docs[:k]

- Hybrid

In [15]:
import inspect

_hsig = inspect.signature(retriever.get_hybrid_docs)
_hparams = _hsig.parameters

def retrieve_hybrid(question: str, k: int):
    if "k" in _hparams:
        docs = retriever.get_hybrid_docs(question, k=k) or []
        return docs[:k]
    docs = retriever.get_hybrid_docs(question) or []
    return docs[:k]

- Rerank

In [16]:
_rsig = inspect.signature(retriever.rerank_docs)
_rparams = _rsig.parameters

def retrieve_rerank(question: str, k: int):
    if "top_n" in _rparams:
        candidate_k = max(60, k * 3) if "candidate_k" in _rparams else None
        kwargs = {"top_n": k}
        if candidate_k is not None:
            kwargs["candidate_k"] = candidate_k
        if "docid_scope_top_n" in _rparams:
            kwargs["docid_scope_top_n"] = None  # 평가용: 필터 OFF
        return retriever.rerank_docs(question, **kwargs) or []
    docs = retriever.rerank_docs(question) or []
    return docs[:k]

### 8. 평가

In [17]:
k_list = [1, 3, 5, 10, 20]   # 원하는 K

runs = [
    ("BM25",   retrieve_bm25),
    ("Dense",  retrieve_dense),
    ("Hybrid", retrieve_hybrid),
    ("Rerank", retrieve_rerank),
]

rows = []
for name, fn in runs:
    res = evaluate_retrieval(dataset, fn, k_list=k_list, debug_n=0, label=name)
    for k in k_list:
        rows.append({
            "Stage": name,
            "K": k,
            "Recall@K": res["Recall"][k],
            "MRR@K": res["MRR"][k],
        })

df = pd.DataFrame(rows).sort_values(["K", "Stage"])
df

[BM25] loaded docs=19387


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embedding

,Stage,K,Recall@K,MRR@K
0,BM25,1,0.100000,0.100000
5,Dense,1,0.133333,0.133333
10,Hybrid,1,0.133333,0.133333
15,Rerank,1,0.066667,0.066667
1,BM25,3,0.133333,0.116667
6,Dense,3,0.166667,0.150000
11,Hybrid,3,0.200000,0.166667
16,Rerank,3,0.066667,0.066667
2,BM25,5,0.166667,0.125000
7,Dense,5,0.166667,0.150000


In [20]:
pivot_recall = df.pivot(index="K", columns="Stage", values="Recall@K")
pivot_mrr    = df.pivot(index="K", columns="Stage", values="MRR@K")

**Recall**

In [21]:
pivot_recall

Stage,BM25,Dense,Hybrid,Rerank
K,,,,
1,0.100000,0.133333,0.133333,0.066667
3,0.133333,0.166667,0.200000,0.066667
5,0.166667,0.166667,0.200000,0.133333
10,0.166667,0.166667,0.233333,0.200000
20,0.200000,0.233333,0.233333,0.266667


**MRR**

In [22]:
pivot_mrr

Stage,BM25,Dense,Hybrid,Rerank
K,,,,
1,0.100000,0.133333,0.133333,0.066667
3,0.116667,0.150000,0.166667,0.066667
5,0.125000,0.150000,0.166667,0.083333
10,0.125000,0.150000,0.172222,0.090000
20,0.128030,0.153704,0.172222,0.093627
